# 📚 Lab 2: Lakebase Postgres with Change Data Feed

## Why This Lab?

In **Lab 1**, you created Delta tables in Unity Catalog - perfect for analytics at scale. But what if your **operational applications** need:
* ⚡ **Low-latency row lookups** (milliseconds, not seconds)
* 🔒 **ACID transactions** for writes
* 🔄 **Real-time updates** flowing back to the data lake

That's where **Lakebase Postgres** comes in!

## What You'll Build:

A **bidirectional data pipeline** connecting:
* 📦 **Unity Catalog Delta** (columnar, analytics-optimized)
* ↔️ **Lakebase Postgres** (row-based, operational-optimized)

### The Magic:

1. **Sync Delta → Postgres** (read-only replicas for fast queries)
2. **App writes to Postgres** (operational tables)
3. **CDF streams back to Delta** (change history for analytics)

You get **the best of both worlds**: fast operational queries + scalable analytics.

## Time to Complete: ~15-20 minutes

**Note:** One manual step required (enabling CDF in the UI) - we'll walk you through it.

---

➡️ **Start by reading the full overview below, then run cells sequentially.**

# Lab 2: Lakebase Postgres with Change Data Feed

## What this lab does:

1. **Creates a Lakebase Postgres project** with a production branch
2. **Syncs Delta tables from Lab1** into Postgres (using Lakebase sync)
3. **Creates operational tables** in Postgres for your application to write to
4. **Enables Change Data Feed (CDF)** to stream changes back to Unity Catalog
5. **Simulates application writes** to test the full bidirectional sync

## Architecture:

```
Unity Catalog (catalog_workshop)
  └─ schema_{your_name} (Lab1 source tables)
       ├─ machines
       ├─ sensor_readings              
       ├─ production_orders             
       └─ maintenance_tickets            
                   ↓ Sync ↓
  Lakebase Postgres (databricks_postgres.public)
       ├─ machines (read-only replica)
       ├─ sensor_readings (read-only)              
       ├─ production_orders (read-only)             
       ├─ maintenance_tickets (read-only)  
       ├─ maintenance_actions (your app writes)
       ├─ work_orders (your app writes)
       ├─ quality_checks (your app writes)
       └─ operator_notes (your app writes)
                   ↓ CDF ↓
  └─ lakebase_{your_name} (CDF history tables + synced tables)
       ├─ machines (synced from Postgres)
       ├─ sensor_readings (synced from Postgres)
       ├─ production_orders (synced from Postgres)
       ├─ maintenance_tickets (synced from Postgres)
       ├─ lb_maintenance_actions_history
       ├─ lb_work_orders_history              
       ├─ lb_quality_checks_history             
       └─ lb_operator_notes_history
```

## Prerequisites:

⚠️ **You must run Lab1 first** to create the source tables:
- `catalog_workshop.schema_{your_name}.machines`
- `catalog_workshop.schema_{your_name}.sensor_readings`
- `catalog_workshop.schema_{your_name}.production_orders`
- `catalog_workshop.schema_{your_name}.maintenance_tickets`

Without these tables, Cell 7 will fail when trying to create synced tables.

## Key Changes from Original:

✅ **Uses schema within catalog_workshop** (not separate catalog)  
✅ **All data in one catalog** for better organization  
✅ **Consistent naming** with Lab1 (`schema_{your_name}`)  
✅ **pipeline_storage schema** for Lakebase sync metadata

## 🏗️ Setting Up Unity Catalog

Before we work with Lakebase, we need to ensure the Unity Catalog structure exists.

### What you'll learn:

* How to create Unity Catalog objects (catalogs and schemas)
* Why we organize data in a **catalog → schema → table** hierarchy
* The `IF NOT EXISTS` pattern for idempotent scripts

### Why check this first?

Lab2 syncs data **from** Unity Catalog **to** Postgres, and then streams changes **back** to Unity Catalog. Both directions require the catalog structure to exist first.

In [0]:
%sql
-- Ensure the Unity Catalog exists (shared catalog for workshop)
-- IF NOT EXISTS makes this safe to re-run without errors
CREATE CATALOG IF NOT EXISTS catalog_workshop;

-- Verify creation by listing all catalogs
-- You should see: main, system, hive_metastore, samples, and catalog_workshop
SHOW CATALOGS;

## 🔄 Python Environment Setup

We need to install specific Python packages to work with Lakebase Postgres.

### Why restart Python?

1. **Clean state**: Ensures no old imports interfere with new packages
2. **Version checks**: We can verify the SDK version before and after upgrade
3. **Kernel stability**: Prevents conflicts between `psycopg-binary` (C extension, crashes) and `psycopg` (pure Python)

### What packages are we installing?

* **databricks-sdk >= 0.118.0** - SDK with Lakebase Postgres support
* **psycopg >= 3.1** - Pure-Python Postgres client (safe for serverless)
* **protobuf < 6.0.0** - Compatibility fix for SDK serialization

### What you'll learn:

* How to safely upgrade packages in a notebook
* Why `psycopg-binary` crashes on serverless compute
* The `dbutils.library.restartPython()` pattern for clean reinstalls

In [0]:
# Restart the Python kernel to start with a clean slate
# This clears all imported modules and variables
dbutils.library.restartPython()

# After restart, check which SDK version is currently installed
import importlib.metadata as md
print(f"databricks-sdk version: {md.version('databricks-sdk')}")
print("\n➡️  Run Cell 3 to upgrade to the required version")

In [0]:
import importlib.metadata as md, subprocess, sys

# Capture the current version before upgrade
try:
    before = md.version("databricks-sdk")
except md.PackageNotFoundError:
    before = None  # Not installed yet

# ─── CRITICAL: Remove psycopg-binary ───
# psycopg-binary uses C extensions that crash on serverless compute
# We use pure-Python psycopg instead (slower but stable)
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "psycopg-binary"])

# ─── Install required packages ───
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade",
                       "databricks-sdk>=0.118.0",  # Lakebase Postgres API support
                       "psycopg>=3.1",              # Pure-Python Postgres client
                       "protobuf<6.0.0"])           # SDK serialization compatibility

# Show version change
after = md.version("databricks-sdk")
print(f"databricks-sdk: {before} -> {after}")
print("\n✅ Packages installed successfully")
print("Restarting Python to unload old psycopg-binary...")

# Restart again to ensure psycopg-binary is fully unloaded
dbutils.library.restartPython()

## 📝 Configuration & Variables

Before creating the Lakebase project, let's define all the naming conventions and locations.

### Why use a slug?

Your email (e.g., `user.name@company.com`) contains special characters that aren't valid in:
* Postgres project names
* Unity Catalog schema names
* Database identifiers

The **slug** removes all non-alphanumeric characters: `user.name@company.com` → `username`

This ensures:
* ✅ Consistency between Lab1 and Lab2
* ✅ Valid identifiers everywhere
* ✅ Easy to read and remember

### Key concepts:

* **PROJECT_BASE** - Base name for Lakebase projects (can create multiple with `-1`, `-2` suffixes)
* **UC_CATALOG** - Unity Catalog where both source and synced data live
* **SCHEMA** - Your source schema from Lab1 (`schema_{your_name}`)
* **LAKEBASE_SCHEMA** - Schema for synced tables and CDF history (`lakebase_{your_name}`)
* **PGDB** - Postgres database name (CDF requires `databricks_postgres`)

### What you'll learn:

* How to create user-specific resource names
* Why we keep all data in one catalog (`catalog_workshop`)
* The difference between source and synced schemas

In [0]:
import re
from databricks.sdk import WorkspaceClient

# Initialize the Databricks SDK workspace client
w = WorkspaceClient()

# Get current user's email and create a slug (remove special chars)
# Example: user.name@company.com -> username
user = w.current_user.me().user_name
slug = re.sub(r"[^a-z0-9]", "", user.split("@")[0].lower())

# ─── Lakebase Project Variables ───
PROJECT_BASE = f"lakebase-ws-{slug}"  # Base name for Lakebase projects
# These are set by Cell 5 after finding/creating a healthy project:
PROJECT    = None  # Actual project name (e.g., lakebase-ws-{your_name}-1)
BRANCH     = None  # Branch path (e.g., projects/.../branches/production)
ENDPOINT   = None  # Endpoint path (e.g., .../endpoints/primary)

# ─── Unity Catalog Variables ───
UC_CATALOG = "catalog_workshop"           # Shared catalog (from Lab 1)
SCHEMA     = f"schema_{slug}"             # Source schema with Lab1 tables
LAKEBASE_SCHEMA = f"lakebase_{slug}"      # Target schema for synced tables + CDF history

# ─── Postgres Variables ───
PGDB       = "databricks_postgres"        # Default database (required for CDF)
host       = None  # Postgres host (resolved in Cell 6)

# Display configuration
print("\n⚙️  Configuration:")
print(f"   PROJECT_BASE     = {PROJECT_BASE}")
print(f"   UC_CATALOG       = {UC_CATALOG}")
print(f"   SCHEMA           = {SCHEMA} (source data from Lab1)")
print(f"   LAKEBASE_SCHEMA  = {LAKEBASE_SCHEMA} (synced tables + CDF history)")
print(f"   PGDB             = {PGDB}")
print(f"\n➡️  Run Cell 5 to create/find Lakebase project")

## 🔌 Creating a Lakebase Postgres Project

Lakebase Postgres is Databricks' **managed PostgreSQL** service that integrates tightly with Unity Catalog.

### What is a Lakebase project?

A project is a **PostgreSQL database cluster** managed by Databricks:
* 💾 **Fully managed** - Databricks handles backups, patches, scaling
* 🔐 **Unity Catalog integration** - Sync Delta tables bidirectionally
* 🚀 **Serverless** - Auto-scales based on load, scale-to-zero when idle
* 🔄 **Change Data Feed** - Stream Postgres writes back to Unity Catalog

### Why the "zombie check" logic?

Sometimes a Lakebase project can get into a bad state where:
* The project exists in the API
* But branches are inaccessible (returns errors)
* This is a "zombie" project - it exists but can't be used

The code below **skips zombies** by:
1. Checking if project exists
2. Testing if branches are accessible
3. If zombie, tries next number (`-2`, `-3`, etc.)
4. Creates a new project if none are healthy

### What you'll learn:

* How to create Lakebase projects via the SDK
* The `projects/{name}/branches/{branch}/endpoints/{endpoint}` path structure
* Why we test for "health" before using a project
* Idempotent patterns (safe to re-run)

In [0]:
# Find or create a healthy Lakebase project
# This logic handles "zombie" projects by checking branch accessibility
from databricks.sdk.service.postgres import Project, ProjectSpec

MAX_ATTEMPTS = 20  # Try up to 10 different project numbers

# Try project names: lakebase-ws-{your_name}-1, -2, -3, etc.
for i in range(1, MAX_ATTEMPTS + 1):
    candidate = f"{PROJECT_BASE}-{i}"
    
    try:
        # Check if project already exists
        w.postgres.get_project(name=f"projects/{candidate}")
        
        # Project exists — verify it's healthy by listing branches
        # Zombie projects fail at this step
        try:
            branches = list(w.postgres.list_branches(parent=f"projects/{candidate}"))
            # Success! This project is healthy and usable
            PROJECT = candidate
            print(f"✅ Project '{candidate}' exists and is healthy ({len(branches)} branch(es))")
            break  # Stop searching, use this one
        except Exception:
            # Project exists but branches are inaccessible (zombie)
            print(f"⚠️  Project '{candidate}' is a zombie (branches inaccessible) — skipping")
            continue  # Try next number
    
    except Exception:
        # Project doesn't exist yet — create it!
        print(f"Creating project '{candidate}'...")
        
        # Create with Postgres 17 and default settings
        op = w.postgres.create_project(
            project=Project(spec=ProjectSpec(
                display_name=candidate,  # Display name in UI
                pg_version=17            # PostgreSQL version
            )),
            project_id=candidate,        # Unique ID (same as display name)
        )
        
        # Wait for creation to complete (async operation)
        result = op.wait()
        PROJECT = candidate
        print(f"✅ Created Lakebase project: {result.name}")
        break  # Done!

else:
    # Loop completed without finding/creating a healthy project
    raise RuntimeError(f"Could not find or create a healthy project after {MAX_ATTEMPTS} attempts")

# ─── Set up dependent variables ───
# Lakebase uses a hierarchical path structure:
# projects/{project}/branches/{branch}/endpoints/{endpoint}
BRANCH   = f"projects/{PROJECT}/branches/production"  # Production branch
ENDPOINT = f"{BRANCH}/endpoints/primary"              # Primary endpoint

print(f"\n✅ Lakebase project ready:")
print(f"   PROJECT  = {PROJECT}")
print(f"   BRANCH   = {BRANCH}")
print(f"   ENDPOINT = {ENDPOINT}")
print(f"\n➡️  Run Cell 6 to get connection details")

## 🔑 Connection Details & Authentication

Now that we have a Lakebase project, we need to get the **connection details** to access it.

### What is an endpoint?

An **endpoint** is the actual Postgres server you connect to:
* Has a **host** (e.g., `xyz.cloud.databricks.com`)
* Listens on port **5432** (standard Postgres)
* Accepts connections over **TLS/SSL**

### How does authentication work?

Lakebase uses **OAuth tokens** instead of passwords:
1. Your Databricks credentials (service principal) generate a short-lived token
2. The token is used as the Postgres password
3. Tokens expire after ~1 hour (automatically refreshed)
4. This is more secure than static passwords

### The `pg_token()` helper function

We create a helper that:
* Fetches a fresh OAuth token on demand
* Uses the SDK's service principal credentials
* Returns the token as a string (used as password)

### What you'll learn:

* How to get endpoint connection details via the SDK
* Why OAuth tokens are better than passwords
* How to create reusable helper functions for authentication

In [0]:
# ─── Get endpoint connection details ───
# Query the endpoint to get its hostname and port
endpoint_info = w.postgres.get_endpoint(name=ENDPOINT)
host = endpoint_info.status.hosts.host  # e.g., xyz.cloud.databricks.com

print(f"✅ Endpoint host: {host}")
print(f"   Port: 5432 (standard Postgres)")
print(f"   Database: {PGDB}")
print(f"   SSL: required")

# ─── Create authentication helper ───
def pg_token():
    """
    Fetch a fresh OAuth token for Postgres authentication.
    
    Lakebase uses OAuth tokens instead of static passwords.
    Tokens expire after ~1 hour and are automatically refreshed.
    
    Returns:
        str: OAuth access token to use as Postgres password
    """
    from databricks.sdk.core import oauth_service_principal
    
    cfg = w.config
    
    # Ensure we're using OAuth M2M (machine-to-machine) authentication
    if not cfg.client_id or not cfg.client_secret:
        raise ValueError(
            "Must use OAuth M2M (client_id + client_secret).\n"
            "Set DATABRICKS_CLIENT_ID and DATABRICKS_CLIENT_SECRET environment variables."
        )
    
    # Get credentials and extract the access token
    creds = oauth_service_principal(cfg)
    return creds.token().access_token

print(f"\n✅ Authentication helper created: pg_token()")
print(f"   Call pg_token() to get a fresh OAuth token for Postgres password")
print(f"\n➡️  Run Cell 7 to create synced tables")

## 🔄 Creating Synced Tables

Now comes the magic: **syncing Delta tables from Unity Catalog into Postgres**.

### What is a synced table?

A synced table is a **read-only replica** of a Unity Catalog Delta table in Postgres:
* 💾 **Source**: Delta table in Unity Catalog (e.g., `catalog_workshop.schema_{your_name}.machines`)
* 🎯 **Destination**: Postgres table (e.g., `databricks_postgres.public.machines`)
* 🔄 **Auto-sync**: Lakebase continuously syncs changes from Delta to Postgres
* 🔒 **Read-only**: Your app can query these tables but **cannot** modify them

### Why sync from Delta to Postgres?

1. **Operational apps** need low-latency row-level access (Postgres)
2. **Analytics** need columnar scans at scale (Delta)
3. **Best of both worlds**: Write analytics in Delta, serve operational queries from Postgres

### What happens in this cell?

1. **Create Unity Catalog schema**:
   - `lakebase_{your_name}` - Holds synced table metadata and CDF history tables
   - The SDK automatically manages internal pipeline metadata

2. **Enable Change Data Feed (CDF)** on source tables:
   - CDF tracks all changes (INSERT/UPDATE/DELETE) to Delta tables
   - Required for continuous/triggered sync modes
   - Adds `_change_type` and `_commit_version` metadata columns

3. **Create synced tables**:
   - One for each Lab1 table: machines, sensor_readings, production_orders, maintenance_tickets
   - Specify primary key columns (required for sync)
   - Choose sync mode: **SNAPSHOT** (one-time full copy), CONTINUOUS (real-time), or TRIGGERED (scheduled)
   - Lakebase handles schema mapping and Postgres table creation automatically

### What you'll learn:

* How to enable Change Data Feed on Delta tables
* How to create synced tables via the SDK with the correct API structure
* Why primary keys are required for sync
* The difference between sync modes:
  - **SNAPSHOT**: One-time full copy (used in this lab)
  - **CONTINUOUS**: Real-time streaming (seconds latency)
  - **TRIGGERED**: Scheduled refresh (cron-based)

In [0]:
from databricks.sdk.service.postgres import (
    SyncedTable,
    SyncedTableSyncedTableSpec,
    SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy,
)

# ─── Step 1: Create Unity Catalog schema ───
# lakebase_{your_name}: Schema for synced table metadata and CDF history tables
#   - Synced tables: Read-only Delta replicas of Postgres tables (registered in UC)
#   - CDF history tables: Capture changes from Postgres operational tables (lb_*_history)
#   - The SDK automatically manages pipeline metadata storage internally
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{LAKEBASE_SCHEMA}")

print(f"✅ Schema created:")
print(f"   {UC_CATALOG}.{LAKEBASE_SCHEMA} (synced tables + CDF history)")

# ─── Step 2: Define primary keys ───
# Synced tables REQUIRE primary keys to track row identity
# Without a PK, Lakebase can't determine which rows changed
PKS = {
    'machines': ['machine_id'],              # Unique machine identifier
    'sensor_readings': ['reading_id'],       # Unique reading ID
    'production_orders': ['order_id'],       # Unique order number
    'maintenance_tickets': ['ticket_id']     # Unique ticket ID
}

# ─── Step 3: Enable Change Data Feed on source tables ───
# CDF tracks all INSERT/UPDATE/DELETE operations
# Adds metadata: _change_type, _commit_version, _commit_timestamp
print(f"\nEnabling Change Data Feed on source tables...")
for tbl in PKS.keys():
    try:
        spark.sql(f"""
            ALTER TABLE {UC_CATALOG}.{SCHEMA}.{tbl} 
            SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"   ✅ {tbl}: CDF enabled")
    except Exception as e:
        # Might already be enabled - not a fatal error
        if "already set" in str(e).lower():
            print(f"   ✅ {tbl}: CDF already enabled")
        else:
            print(f"   ⚠️  {tbl}: Could not enable CDF - {str(e)[:80]}")

# ─── Step 4: Create synced tables ───
# These are read-only Postgres replicas of your Delta tables
print(f"\nCreating synced tables (Delta → Postgres)...")
for tbl, pk in PKS.items():
    # The synced table's Unity Catalog identifier (for metadata)
    synced_table_id = f"{UC_CATALOG}.{LAKEBASE_SCHEMA}.{tbl}"
    
    try:
        # Create the synced table using the correct SDK API
        w.postgres.create_synced_table(
            synced_table=SyncedTable(
                spec=SyncedTableSyncedTableSpec(
                    # Source: Delta table in Unity Catalog
                    source_table_full_name=f"{UC_CATALOG}.{SCHEMA}.{tbl}",
                    
                    # Which Lakebase branch to sync to
                    branch=BRANCH,  # projects/{project}/branches/production
                    
                    # REQUIRED: Primary key columns (must be a list)
                    primary_key_columns=pk,
                    
                    # Sync mode: SNAPSHOT for one-time full copy
                    # Alternatives: CONTINUOUS (real-time), TRIGGERED (scheduled)
                    scheduling_policy=SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.SNAPSHOT,
                    
                    # Destination Postgres database
                    postgres_database=PGDB,  # databricks_postgres
                    
                    # Auto-create schema and table in Postgres if missing
                    create_database_objects_if_missing=True,
                )
            ),
            # Unity Catalog identifier for the synced table metadata
            synced_table_id=synced_table_id,
        ).wait()  # Wait for sync to provision (may take 1-2 minutes)
        
        print(f"   ✅ {tbl:30s} synced (PK: {', '.join(pk)})")
    except Exception as e:
        # Already exists, or permissions issue
        if "already exists" in str(e).lower():
            print(f"   ✅ {tbl:30s} already synced")
        else:
            print(f"   ❌ {tbl:30s} {e.__class__.__name__} — {str(e)[:80]}")

print(f"\n✅ Synced tables created! Postgres now has read-only replicas.")
print(f"   Snapshot complete - tables copied once from Delta to Postgres.")
print(f"\n➡️  Run Cell 8 to verify the schemas were created")

In [0]:
# ─── Verify Unity Catalog structure ───
# Check that the schema and synced tables were created successfully

print(f"Checking Unity Catalog structure...\n")

# Get all schemas in the catalog
schemas = spark.sql(f"SHOW SCHEMAS IN {UC_CATALOG}").collect()
schema_names = [s.databaseName for s in schemas]

# Check if our Lakebase schema exists
if LAKEBASE_SCHEMA in schema_names:
    print(f"✅ Schema {UC_CATALOG}.{LAKEBASE_SCHEMA} exists")
    
    # List tables in the schema
    tables = spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{LAKEBASE_SCHEMA}").collect()
    
    if tables:
        print(f"\n📊 Synced tables in {UC_CATALOG}.{LAKEBASE_SCHEMA}:")
        for t in tables:
            # Get row count for each table
            try:
                count = spark.table(f"{UC_CATALOG}.{LAKEBASE_SCHEMA}.{t.tableName}").count()
                print(f"   • {t.tableName:30s} ({count:,} rows)")
            except:
                print(f"   • {t.tableName:30s} (provisioning...)")
    else:
        print(f"\n⚠️  No tables yet - synced tables may still be provisioning")
        print(f"   Wait 1-2 minutes and check the Lakebase UI for sync status")
else:
    print(f"❌ Schema {UC_CATALOG}.{LAKEBASE_SCHEMA} not found")
    print(f"   Cell 7 may have failed - check error messages above")

print(f"\n➡️  Run Cell 9 to create operational tables in Postgres")

In [0]:
import psycopg

print("="*70)
print("🔍 SYNCED TABLE STATUS CHECK")
print("="*70)

# Expected synced tables
expected_tables = ['machines', 'sensor_readings', 'production_orders', 'maintenance_tickets']

print("\n[1] Checking sync status via SDK...\n")
for tbl in expected_tables:
    synced_table_id = f"{UC_CATALOG}.{LAKEBASE_SCHEMA}.{tbl}"
    try:
        # Get the synced table status from the SDK
        st = w.postgres.get_synced_table(name=f"synced_tables/{synced_table_id}")
        # Handle different SDK response structures
        if hasattr(st.status, 'current_state'):
            state = st.status.current_state.value if hasattr(st.status.current_state, 'value') else str(st.status.current_state)
        elif hasattr(st, 'state'):
            state = st.state.value if hasattr(st.state, 'value') else str(st.state)
        else:
            # Fallback: show the full status object
            state = str(st.status) if st.status else "UNKNOWN"
        print(f"  {tbl:30s} → {state}")
    except Exception as e:
        print(f"  {tbl:30s} → ❌ Not found or error: {str(e)[:50]}")

print("\n[2] Checking actual Postgres tables...\n")
try:
    with psycopg.connect(
        host=host,
        port=5432,
        dbname=PGDB,
        user=user,
        password=w.postgres.generate_database_credential(endpoint=ENDPOINT).token,
        sslmode="require",
        autocommit=True  # Prevent transaction abort on errors
    ) as conn:
        for tbl in expected_tables:
            try:
                # Try to query the table
                result = conn.execute(f"SELECT COUNT(*) FROM public.{tbl}").fetchone()
                count = result[0] if result else 0
                print(f"  ✅ {tbl:30s} → {count:,} rows")
            except Exception as e:
                error_msg = str(e)
                if "does not exist" in error_msg:
                    print(f"  ⏳ {tbl:30s} → Table not created yet (still provisioning)")
                else:
                    print(f"  ❌ {tbl:30s} → Error: {error_msg[:50]}")
except Exception as e:
    print(f"  ❌ Connection failed: {str(e)[:100]}")

print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)
print("• ACTIVE + rows > 0     → ✅ Fully synced and ready")
print("• ACTIVE + rows = 0     → ⚠️  Sync pipeline running, data coming soon")
print("• PROVISIONING          → ⏳ Initial setup, wait 1-2 minutes")
print("• Table not found       → ⏳ Postgres table creation in progress")
print("="*70)

## 📝 Creating Operational Tables

Now we'll create **application-owned tables** that your operational app writes to.

### What's the difference?

| Type | Source | Access | Use Case |
|------|--------|--------|----------|
| **Synced tables** | Unity Catalog Delta | Read-only | Query analytics data from operational apps |
| **Operational tables** | Postgres native | Read-write | App transactions (orders, maintenance, notes) |

### Why create separate operational tables?

1. **Write access**: Synced tables are read-only. Your app needs writable tables.
2. **Operational transactions**: Fast row-level INSERT/UPDATE/DELETE in Postgres
3. **CDF capture**: Changes to these tables stream back to Unity Catalog via CDF
4. **Separation of concerns**: Analytics data (synced) vs. operational data (native)

### What tables are we creating?

1. **maintenance_actions** - Actions taken by technicians
2. **work_orders** - Scheduled maintenance work
3. **quality_checks** - QA inspection results
4. **operator_notes** - Shift handoff notes and observations

All tables have **foreign keys** linking back to:
* `machine_id` → machines (from synced tables)
* `ticket_id` → maintenance_tickets (from synced tables)
* `order_id` → production_orders (from synced tables)

### ⚠️ Critical: REPLICA IDENTITY FULL

Cell 9 also sets `REPLICA IDENTITY FULL` on all operational tables. **This is required for CDF!**

Without it:
* ❌ CDF will **skip** these tables during sync
* ❌ Changes won't stream to Unity Catalog
* ❌ You'll see "Tables without REPLICA IDENTITY FULL will be skipped" warnings

What it does:
* ✅ Tells Postgres to include **all column values** in the Write-Ahead Log (WAL)
* ✅ Enables CDF to capture the full before/after state of each row
* ✅ Required for UPDATE and DELETE change tracking

### What you'll learn:

* How to execute SQL DDL via `psycopg`
* Postgres data types (`SERIAL`, `TIMESTAMPTZ`, `VARCHAR`, `TEXT`)
* Why we use `CREATE TABLE IF NOT EXISTS` (idempotent)
* Foreign key patterns for relational data
* **REPLICA IDENTITY FULL** - Critical requirement for CDF to capture changes

In [0]:
import psycopg

# Connect to Postgres using OAuth token authentication
with psycopg.connect(
    host=host,              # From Cell 6
    port=5432,              # Standard Postgres port
    dbname=PGDB,            # databricks_postgres
    user=user,              # Your email (Databricks identity)
    password=w.postgres.generate_database_credential(endpoint=ENDPOINT).token,    # OAuth token (NOT a static password)
    sslmode="require",      # Enforce TLS encryption
    autocommit=True         # Each statement commits immediately
) as conn:
    print("🛠️  Creating operational tables (your app will write to these)...\n")
    
    # ─── TABLE 1: maintenance_actions ───
    # Tracks actions taken by technicians (repairs, inspections, preventive maintenance)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.maintenance_actions (
            action_id SERIAL PRIMARY KEY,              -- Auto-incrementing ID
            machine_id INTEGER NOT NULL,                -- FK to machines table
            ticket_id INTEGER,                          -- FK to maintenance_tickets (optional)
            action_type VARCHAR(50),                    -- preventive, corrective, inspection
            description TEXT,                           -- What was done
            performed_by VARCHAR(100),                  -- Technician name
            performed_at TIMESTAMPTZ DEFAULT now(),     -- When started (auto-timestamp)
            status VARCHAR(50),                         -- in_progress, completed, failed
            completed_at TIMESTAMPTZ                    -- When finished (nullable)
        );
    """)
    print("✅ maintenance_actions     (tracks repair and inspection work)")
    
    # ─── TABLE 2: work_orders ───
    # Work orders scheduled for machines (preventive or reactive)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.work_orders (
            work_order_id SERIAL PRIMARY KEY,          -- Auto-incrementing ID
            machine_id INTEGER NOT NULL,                -- FK to machines table
            order_id INTEGER,                           -- FK to production_orders (optional)
            priority VARCHAR(20),                       -- low, medium, high, critical
            title VARCHAR(200),                         -- Short description
            description TEXT,                           -- Detailed work instructions
            assigned_to VARCHAR(100),                   -- Technician assigned
            created_at TIMESTAMPTZ DEFAULT now(),       -- When created (auto-timestamp)
            due_date DATE,                              -- Target completion date
            status VARCHAR(50)                          -- open, assigned, in_progress, closed
        );
    """)
    print("✅ work_orders            (scheduled maintenance tasks)")
    
    # ─── TABLE 3: quality_checks ───
    # QA inspection results for production orders
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.quality_checks (
            check_id SERIAL PRIMARY KEY,               -- Auto-incrementing ID
            order_id INTEGER,                           -- FK to production_orders
            machine_id INTEGER,                         -- FK to machines table
            check_type VARCHAR(50),                     -- dimensional, functional, visual
            result VARCHAR(20),                         -- pass, fail, conditional
            defect_code VARCHAR(50),                    -- Defect classification (if failed)
            notes TEXT,                                 -- Inspector observations
            inspector VARCHAR(100),                     -- QA inspector name
            checked_at TIMESTAMPTZ DEFAULT now()        -- Inspection timestamp
        );
    """)
    print("✅ quality_checks         (QA inspection results)")
    
    # ─── TABLE 4: operator_notes ───
    # Operator observations, alerts, and shift handoff notes
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.operator_notes (
            note_id SERIAL PRIMARY KEY,                -- Auto-incrementing ID
            machine_id INTEGER,                         -- FK to machines (optional)
            ticket_id INTEGER,                          -- FK to maintenance_tickets (optional)
            order_id INTEGER,                           -- FK to production_orders (optional)
            note_type VARCHAR(50),                      -- alert, handoff, resolution, observation
            content TEXT,                               -- Free-form note text
            created_by VARCHAR(100),                    -- Operator name
            created_at TIMESTAMPTZ DEFAULT now()        -- Note timestamp
        );
    """)
    print("✅ operator_notes         (operator observations and handoffs)")
    
    # ─── CRITICAL: Enable REPLICA IDENTITY FULL for CDF ───
    # Without this, CDF will SKIP these tables!
    # REPLICA IDENTITY FULL tells Postgres to include all column values in the WAL (Write-Ahead Log)
    # This is required for CDF to capture the full before/after state of each row
    print("\n🔧 Enabling REPLICA IDENTITY FULL (required for CDF)...\n")
    
    for table in ['maintenance_actions', 'work_orders', 'quality_checks', 'operator_notes']:
        conn.execute(f"ALTER TABLE public.{table} REPLICA IDENTITY FULL;")
        print(f"✅ {table:25s} → REPLICA IDENTITY FULL enabled")
    
print("\n✅ All operational tables created in Postgres!")
print(f"   Tables are ready for INSERT/UPDATE/DELETE from your app.")
print(f"   REPLICA IDENTITY FULL is enabled — CDF will capture all changes.")
print(f"\n➡️  Next: Complete the manual CDF setup in Cell 10")

In [0]:
import psycopg

# Connect and list all tables in the public schema
with psycopg.connect(
    host=host,
    port=5432,
    dbname=PGDB,
    user=user,
    password=w.postgres.generate_database_credential(endpoint=ENDPOINT).token,
    sslmode="require"
) as conn:
    result = conn.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        AND table_type = 'BASE TABLE'
        ORDER BY table_name;
    """).fetchall()
    
    print(f"📊 Tables in {PROJECT} → databricks_postgres.public:\n")
    
    # Expected tables
    synced_expected = {'machines', 'sensor_readings', 'production_orders', 'maintenance_tickets'}
    operational_expected = {'maintenance_actions', 'work_orders', 'quality_checks', 'operator_notes'}
    
    found_tables = {row[0] for row in result}
    
    synced_found = found_tables & synced_expected
    operational_found = found_tables & operational_expected
    
    if synced_found:
        print("🔄 Synced tables (from Unity Catalog):")
        for t in sorted(synced_found):
            print(f"   ✅ {t}")
        missing_synced = synced_expected - synced_found
        if missing_synced:
            for t in sorted(missing_synced):
                print(f"   ❌ {t} (MISSING)")
    
    if operational_found:
        print("\n📝 Operational tables (app-writable):")
        for t in sorted(operational_found):
            print(f"   ✅ {t}")
        missing_operational = operational_expected - operational_found
        if missing_operational:
            for t in sorted(missing_operational):
                print(f"   ❌ {t} (MISSING)")
    else:
        print("\n❌ NO OPERATIONAL TABLES FOUND!")
        print("\n   This means Cell 9 either:")
        print("   1. Hasn't been run yet, OR")
        print("   2. Was run against a different project")
        print("\n   Solution: Re-run Cell 9 to create the operational tables.")
    
    print(f"\n📊 Total user tables: {len(found_tables)}")

## ⚠️ MANUAL STEP - Configure Change Data Feed

Stop here and complete this before running Cell 11. The SDK does not yet expose a method to start CDF.

### Pre-requisite:

⚠️ **Cell 9 must be run first!** It creates the operational tables AND sets `REPLICA IDENTITY FULL` on them.

Without `REPLICA IDENTITY FULL`, CDF will show: "Tables without REPLICA IDENTITY FULL will be skipped."

### Steps:

1. **Open Lakebase Postgres** from the app switcher (top right)
2. **Select your project** (e.g. `lakebase-ws-{your_name}-1`) → branch `production`
3. **Go to the Change Data Feed tab** → click **Start**
4. **Configure**:
   - **Database**: `databricks_postgres`
   - **Schema**: `public`
   - **To Catalog**: `catalog_workshop`
   - **Schema**: `lakebase_{your_name}` (where CDF history tables will land)
5. **Review the table list** - you should see all 4 operational tables (maintenance_actions, work_orders, quality_checks, operator_notes)
6. **Click Start**

### What this does:

CDF will capture all INSERT/UPDATE/DELETE operations from your Postgres operational tables and stream them as Delta tables:
- `lb_maintenance_actions_history`
- `lb_work_orders_history`
- `lb_quality_checks_history`
- `lb_operator_notes_history`

These history tables will be created in `catalog_workshop.lakebase_{your_name}`.

## 🧪 Testing the Data Pipeline

Now let's **simulate an operational app** writing to Postgres and verify that changes flow back to Unity Catalog via CDF.

### What are we testing?

The complete **bidirectional data flow**:

```
   Lab1 Source Tables
         ↓
   Unity Catalog Delta
         ↓ (sync)
   Postgres Synced Tables (read-only)
         👇
   Postgres Operational Tables (read-write) ← Your App Writes Here
         ↓ (CDF)
   Unity Catalog History Tables (change events)
```

### What operations will we test?

1. **INSERT** - Add new records (maintenance actions, work orders, quality checks, notes)
2. **UPDATE** - Modify existing records (change status, mark completed)
3. **DELETE** - Remove records (cancel work orders)

CDF captures **all three change types** and streams them to Unity Catalog.

### What happens after we run this?

1. Data is written to Postgres operational tables
2. **Wait ~30 seconds** for CDF to process
3. Check Unity Catalog for new CDF history tables:
   - `lb_maintenance_actions_history`
   - `lb_work_orders_history`
   - `lb_quality_checks_history`
   - `lb_operator_notes_history`

### What you'll learn:

* How to execute DML (INSERT/UPDATE/DELETE) via `psycopg`
* Realistic operational data patterns
* How CDF captures change events
* The `_change_type` column (insert, update_postimage, delete)

In [0]:
# ─── Simulate operational app writes to test CDF sync ───
# Performs INSERTs, UPDATEs, and DELETEs so CDF captures all change types.
# This is what a real manufacturing app would do when operators interact with it.

import psycopg
from datetime import date, timedelta

# Connect to Postgres
with psycopg.connect(
    host=host, port=5432, dbname=PGDB, user=user,
    password=pg_token(), sslmode="require", autocommit=True
) as conn:

    # ─── STEP 1: INSERTs (Create New Records) ───
    print("📝 Inserting sample operational data...\n")

    # maintenance_actions - Technician work log
    conn.execute("""
        INSERT INTO public.maintenance_actions (machine_id, ticket_id, action_type, description, performed_by, status)
        VALUES
            (1, 101, 'preventive', 'Replaced hydraulic fluid and filters', 'operator_jones', 'completed'),
            (2, NULL, 'inspection', 'Routine vibration analysis on spindle', 'tech_smith', 'completed'),
            (3, 205, 'corrective', 'Replaced worn bearing on conveyor belt', 'tech_martinez', 'in_progress'),
            (1, 102, 'preventive', 'Calibrated temperature sensors', 'operator_jones', 'completed');
    """)
    print("✅ maintenance_actions:  4 rows inserted")

    # work_orders - Scheduled maintenance work
    conn.execute(f"""
        INSERT INTO public.work_orders (machine_id, order_id, priority, title, description, assigned_to, due_date, status)
        VALUES
            (2, 1001, 'high', 'Spindle replacement needed', 'Vibration exceeded threshold by 40%%', 'tech_smith', '{date.today() + timedelta(days=2)}', 'assigned'),
            (3, 1002, 'critical', 'Conveyor emergency stop triggered', 'Belt misalignment detected by sensor', 'tech_martinez', '{date.today()}', 'in_progress'),
            (1, 1003, 'low', 'Scheduled lubrication', 'Monthly lubrication per maintenance plan', 'operator_jones', '{date.today() + timedelta(days=7)}', 'open');
    """)
    print("✅ work_orders:          3 rows inserted")

    # quality_checks - QA inspection results
    conn.execute("""
        INSERT INTO public.quality_checks (order_id, machine_id, check_type, result, defect_code, notes, inspector)
        VALUES
            (1001, 2, 'dimensional', 'pass', NULL, 'All tolerances within spec', 'qc_nguyen'),
            (1001, 2, 'functional', 'fail', 'VIB-003', 'Excessive vibration at 3000rpm', 'qc_nguyen'),
            (1002, 3, 'visual', 'conditional', 'ALN-001', 'Minor belt wear observed, monitor closely', 'qc_park'),
            (1003, 1, 'functional', 'pass', NULL, 'Operating within normal parameters', 'qc_nguyen');
    """)
    print("✅ quality_checks:       4 rows inserted")

    # operator_notes - Shift notes and observations
    conn.execute("""
        INSERT INTO public.operator_notes (machine_id, ticket_id, order_id, note_type, content, created_by)
        VALUES
            (2, NULL, 1001, 'alert', 'Hearing unusual grinding noise from spindle during high-speed operation', 'operator_jones'),
            (3, 205, 1002, 'handoff', 'Shift handoff: conveyor belt realignment in progress, do not restart', 'tech_martinez'),
            (1, 102, NULL, 'resolution', 'Temperature sensors recalibrated, readings now within 0.5C tolerance', 'operator_jones');
    """)
    print("✅ operator_notes:       3 rows inserted")

    # ─── STEP 2: UPDATEs (Modify Existing Records) ───
    print("\n🔄 Simulating status updates (what happens when work is completed)...\n")

    # Mark maintenance action as completed
    conn.execute("""
        UPDATE public.maintenance_actions
        SET status = 'completed', completed_at = now()
        WHERE machine_id = 3 AND status = 'in_progress';
    """)
    print("✅ maintenance_actions:  Updated machine 3 action → completed")

    # Close work order
    conn.execute("""
        UPDATE public.work_orders
        SET status = 'closed'
        WHERE machine_id = 2 AND priority = 'high';
    """)
    print("✅ work_orders:          Updated spindle replacement → closed")

    # ─── STEP 3: DELETE (Remove Records) ───
    print("\n🗑️  Simulating a cancellation (DELETE operation)...\n")

    # Cancel low-priority work order
    conn.execute("""
        DELETE FROM public.work_orders
        WHERE priority = 'low' AND status = 'open';
    """)
    print("✅ work_orders:          Deleted 1 row (low priority, cancelled)")

print("\n" + "="*70)
print("✅ All operational writes complete!")
print("="*70)
print("\n🕒 Wait ~30 seconds for CDF to process these changes...")
print("\n📊 CDF will create history tables in Unity Catalog:")
print("   • lb_maintenance_actions_history")
print("   • lb_work_orders_history")
print("   • lb_quality_checks_history")
print("   • lb_operator_notes_history")
print("\n➡️  Run Cell 12 to verify the end-to-end pipeline")

## ✅ End-to-End Verification

Let's verify the **complete bidirectional data flow** from Lab1 all the way through to CDF history tables.

### What this cell checks:

1. **Unity Catalog Structure**
   - Source schema (`schema_{your_name}`) with Lab1 tables
   - Target schema (`lakebase_{your_name}`) with synced tables and CDF history
   - Row counts for all tables

2. **Lakebase Project**
   - Project, branch, and endpoint details
   - Connection host and database

3. **Postgres Tables**
   - Synced tables (read-only replicas from Unity Catalog)
   - Operational tables (app-owned, writable)
   - Row counts for each table

4. **Change Data Feed Status**
   - CDF history tables (`lb_*_history`)
   - Change event counts
   - Verification that CDF is streaming data

### Success criteria:

✅ All 4 Lab1 source tables exist  
✅ All 4 synced tables exist in Postgres  
✅ All 4 operational tables exist in Postgres  
✅ CDF history tables exist with change events  

### What you'll learn:

* How to verify complex data pipelines
* Reading Postgres system catalogs (`pg_tables`)
* Checking CDF table naming conventions
* End-to-end data lineage verification

In [0]:
# ─── Comprehensive status check ───
# Verifies the complete data pipeline from Lab1 source tables
# through Postgres synced/operational tables to CDF history tables

import psycopg

print("="*80)
print("🔍 LAB 2 - FINAL STATUS CHECK")
print("="*80)
print("\nVerifying the complete bidirectional data pipeline...")

# 1. Unity Catalog structure
print("\n[1] Unity Catalog Structure:")
print(f"\n  ✅ Catalog: {UC_CATALOG}")

try:
    schemas = spark.sql(f"SHOW SCHEMAS IN {UC_CATALOG}").collect()
    for s in schemas:
        schema_name = s.databaseName
        if schema_name == SCHEMA:
            print(f"    ├─ {SCHEMA} (Lab1 source tables)")
            try:
                tables = spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{SCHEMA}").collect()
                if tables:
                    print(f"      └─ {len(tables)} table(s): {', '.join([t.tableName for t in tables[:4]])}")
                else:
                    print(f"      └─ ❌ NO TABLES - Run Lab1 first!")
            except:
                print(f"      └─ ❌ Schema doesn't exist yet - Run Lab1 first!")
        elif schema_name == LAKEBASE_SCHEMA:
            print(f"    ├─ {LAKEBASE_SCHEMA} (synced tables + CDF history)")
            try:
                tables = spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{LAKEBASE_SCHEMA}").collect()
                if tables:
                    print(f"      └─ {len(tables)} table(s): {', '.join([t.tableName for t in tables])}")
                else:
                    print(f"      └─ (empty - synced tables may still be provisioning)")
            except:
                print(f"      └─ (not accessible)")
        elif schema_name == 'pipeline_storage':
            print(f"    ├─ pipeline_storage (sync metadata)")
except Exception as e:
    print(f"  ❌ Error: {str(e)[:100]}")

# 2. Lakebase Project
print(f"\n[2] Lakebase Postgres Project:")
if PROJECT:
    print(f"  ✅ Project: {PROJECT}")
    print(f"  ✅ Branch: {BRANCH}")
    print(f"  ✅ Endpoint: {ENDPOINT}")
    print(f"  ✅ Host: {host}")
    print(f"  ✅ Database: {PGDB}")
else:
    print(f"  ❌ Not created yet - run Cell 5")

# 3. Postgres Tables
if host:
    print(f"\n[3] Postgres Tables (public schema):")
    try:
        with psycopg.connect(
            host=host, port=5432, dbname=PGDB, user=user,
            password=pg_token(), sslmode="require", autocommit=True
        ) as conn:
            # Synced tables (read-only)
            synced = conn.execute("""
                SELECT tablename FROM pg_tables 
                WHERE schemaname = 'public' 
                  AND tablename IN ('machines', 'sensor_readings', 'production_orders', 'maintenance_tickets')
                ORDER BY tablename;
            """).fetchall()
            print(f"\n  Synced tables (read-only from UC):")
            if synced:
                for (tbl,) in synced:
                    count = conn.execute(f"SELECT COUNT(*) FROM public.{tbl}").fetchone()[0]
                    print(f"    ✅ {tbl:30s} ({count} rows)")
            else:
                print(f"    ❌ None synced yet - check sync status")
            
            # Operational tables (your app writes)
            operational = conn.execute("""
                SELECT tablename FROM pg_tables 
                WHERE schemaname = 'public' 
                  AND tablename IN ('maintenance_actions', 'work_orders', 'quality_checks', 'operator_notes')
                ORDER BY tablename;
            """).fetchall()
            print(f"\n  Operational tables (app-owned, CDF-enabled):")
            if operational:
                for (tbl,) in operational:
                    count = conn.execute(f"SELECT COUNT(*) FROM public.{tbl}").fetchone()[0]
                    print(f"    ✅ {tbl:30s} ({count} rows)")
            else:
                print(f"    ❌ None created yet - run Cell 9")
    except Exception as e:
        print(f"  ❌ Could not connect: {str(e)[:100]}")

# 4. CDF Status
print(f"\n[4] Change Data Feed Status:")
try:
    cdf_tables = spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{LAKEBASE_SCHEMA}").collect()
    history_tables = [t.tableName for t in cdf_tables if t.tableName.startswith('lb_') and t.tableName.endswith('_history')]
    if history_tables:
        print(f"  ✅ CDF is running! Found {len(history_tables)} history table(s):")
        for tbl in history_tables:
            count = spark.sql(f"SELECT COUNT(*) FROM {UC_CATALOG}.{LAKEBASE_SCHEMA}.{tbl}").collect()[0][0]
            print(f"    ✅ {tbl:40s} ({count} change events)")
    else:
        print(f"  ❌ No CDF history tables yet")
        print(f"     ➡️  Complete the manual step in Cell 10 to enable CDF")
except Exception as e:
    print(f"  ❌ Could not check: {str(e)[:100]}")

print("\n" + "="*80)
print("Next Steps:")
print("="*80)
print("1. ⚠️  If source tables are missing: Open Lab1 and run it first")
print("2. ✅ If synced tables exist: Complete manual CDF setup (Cell 10)")
print("3. ✅ If CDF is running: Check history tables for change events")
print("="*80)

## 🎉 Lab 2 Complete! 

### What You Accomplished:

✅ **Created a Lakebase Postgres project** with managed PostgreSQL  
✅ **Synced Delta tables to Postgres** for low-latency operational access  
✅ **Created operational tables** for application writes  
✅ **Enabled Change Data Feed** to stream changes back to Unity Catalog  
✅ **Tested the complete pipeline** with INSERT/UPDATE/DELETE operations  

### Key Concepts Learned:

1. **Bidirectional Sync**
   - Delta → Postgres: Synced tables (analytics to operational)
   - Postgres → Delta: Change Data Feed (operational to analytics)

2. **Lakebase Postgres**
   - Managed PostgreSQL with Unity Catalog integration
   - OAuth token authentication
   - Project/branch/endpoint hierarchy

3. **Change Data Feed (CDF)**
   - Captures INSERT/UPDATE/DELETE operations
   - Creates history tables with `_change_type` metadata
   - Enables time-travel and audit trails

4. **Data Architecture Patterns**
   - Read-only synced tables vs. writable operational tables
   - Primary keys for sync requirements
   - Foreign key relationships across systems

### What's Next?

💡 **Explore the data:**
* Query CDF history tables to see change events
* Join synced tables with operational tables
* Build dashboards on top of the unified data

🚀 **Build a real app:**
* Create a Flask/FastAPI app that writes to operational tables
* Use the synced tables for read queries
* Monitor CDF for real-time change notifications

🧹 **Clean up:**
* Run the [Cleanup notebook](#notebook-3808656171941983) to remove all resources
* Or keep the project running for further experimentation

### Additional Resources:

* [Lakebase Postgres Documentation](https://docs.databricks.com)
* [Change Data Feed Guide](https://docs.databricks.com/delta/delta-change-data-feed.html)
* [Unity Catalog Best Practices](https://docs.databricks.com/data-governance/unity-catalog/index.html)